In [22]:
# Cell 1 — Imports & config
import pandas as pd
import numpy as np
from pathlib import Path

# ---- Paths (edit if needed) ----
PATH_IN  = Path("MS_Capstone_Final_Clean.xlsx")   # your latest input
PATH_OUT = Path("MS_Capstone_Final_Quality_Check_Summary.xlsx") # output workbook

# Display options (safe defaults)
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print(f"Input file:  {PATH_IN}")
print(f"Output file: {PATH_OUT}")


Input file:  MS_Capstone_Final_Clean.xlsx
Output file: MS_Capstone_Final_Quality_Check_Summary.xlsx


In [23]:
# Cell 2 — Load workbook and pick the primary sheet
xls = pd.ExcelFile(PATH_IN)
print("Sheets found:", xls.sheet_names)

# If your consolidated table is NOT the first sheet, set its name here:
PRIMARY_SHEET_NAME = xls.sheet_names[0]

df_raw = pd.read_excel(PATH_IN, sheet_name=PRIMARY_SHEET_NAME, dtype=str)  # read as strings first
print(f"Loaded sheet: '{PRIMARY_SHEET_NAME}' -> shape={df_raw.shape}")
print("Columns:", list(df_raw.columns))
df_raw.head()


Sheets found: ['Raw Data', 'Label Description']
Loaded sheet: 'Raw Data' -> shape=(585, 32)
Columns: ['Company', 'Ticker', 'AV_Ticker.BSE', 'Sector', 'Cap Category', 'Period', 'Sales', 'Expenses', 'Operating Profit', 'OPM %', 'Other Income', 'Interest', 'Depreciation', 'Profit before tax', 'Tax %', 'Net Profit', 'EPS in Rs', 'Result Date', 'Revenue', 'Financing Profit', 'Financing Margin %', '_T_date', 'Closing Price (T)', '_T1_date', 'Closing Price (T+1)', '_T2_date', 'Closing Price (T+2)', '_T3_date', 'Closing Price (T+3)', '3-Day Avg Price Post Result', '3-Day Return (%)', 'Movement Label (1,0)']


,Company,Ticker,AV_Ticker.BSE,Sector,Cap Category,Period,Sales,Expenses,Operating Profit,OPM %,Other Income,Interest,Depreciation,Profit before tax,Tax %,Net Profit,EPS in Rs,Result Date,Revenue,Financing Profit,Financing Margin %,_T_date,Closing Price (T),_T1_date,Closing Price (T+1),_T2_date,Closing Price (T+2),_T3_date,Closing Price (T+3),3-Day Avg Price Post Result,3-Day Return (%),"Movement Label (1,0)"
0,Tata Consultancy Services,TCS,TCS.BSE,IT / Technology,Large,Jun 2022,52758,39342,13416,25,789,199,1230,12776,25,9519,25.9,08-07-2022,NaN,NaN,NaN,2022-07-08,3265.45,2022-07-11,3113.8,2022-07-12,3084.7,2022-07-13,3038.75,3079.0833333333335,-5.71,0
1,Tata Consultancy Services,TCS,TCS.BSE,IT / Technology,Large,Sep 2022,55309,40793,14516,26,965,148,1237,14096,26,10465,28.51,10-10-2022,NaN,NaN,NaN,2022-10-10,3118.55,2022-10-11,3069.55,2022-10-12,3100.75,2022-10-13,3103.3,3091.2000000000003,-0.88,0
2,Tata Consultancy Services,TCS,TCS.BSE,IT / Technology,Large,Dec 2022,58229,42676,15553,27,520,160,1269,14644,26,10883,29.64,09-01-2023,NaN,NaN,NaN,2023-01-09,3319.95,2023-01-10,3286.4,2023-01-11,3328.7,2023-01-12,3334.35,3316.4833333333336,-0.1,0
3,Tata Consultancy Services,TCS,TCS.BSE,IT / Technology,Large,Mar 2023,59162,43388,15774,27,1175,272,1286,15391,26,11436,31.13,12-04-2023,NaN,NaN,NaN,2023-04-12,3241.65,2023-04-13,3188.85,2023-04-17,3139.5,2023-04-18,3130.75,3153.0333333333333,-2.73,0
4,Tata Consultancy Services,TCS,TCS.BSE,IT / Technology,Large,Jun 2023,59381,44383,14998,25,1397,163,1243,14989,26,11120,30.26,12-07-2023,NaN,NaN,NaN,2023-07-12,3259.9,2023-07-13,3340.55,2023-07-14,3514.65,2023-07-17,3491.7,3448.966666666667,5.8,1


In [24]:
# Cell 3 — Robust date parsing (replacement)

from datetime import datetime

df = df_raw.copy()

# 1) Trim whitespace and standardize empty markers
for c in df.columns:
    if df[c].dtype == object:
        df[c] = (df[c].astype(str)
                        .str.strip()
                        .replace({"": np.nan, "None": np.nan, "none": np.nan, "NaN": np.nan, "nan": np.nan, "—": np.nan, "–": np.nan}))

# 2) Helper: parse many real-world date shapes (incl. Excel serials)
EPOCH_1900 = pd.Timestamp("1899-12-30")  # Excel 1900 system base

def parse_one_date(x):
    if pd.isna(x):
        return pd.NaT

    # Already datetime-like
    if isinstance(x, (pd.Timestamp, datetime, np.datetime64)):
        return pd.to_datetime(x, errors="coerce")

    # Excel serial numbers (allow ints/floats, ignore booleans)
    if isinstance(x, (int, float)) and not isinstance(x, bool) and np.isfinite(x):
        try:
            ts = EPOCH_1900 + pd.to_timedelta(int(x), unit="D")
            # sanity bounds to avoid parsing random numerics as dates
            if pd.Timestamp("1990-01-01") <= ts <= pd.Timestamp("2035-12-31"):
                return ts
        except Exception:
            pass

    # Strings: normalize separators
    s = str(x).strip()
    if s == "":
        return pd.NaT
    s2 = s.replace(".", "-").replace("/", "-")

    # Try day-first first (works for India formats)
    d = pd.to_datetime(s2, errors="coerce", dayfirst=True, infer_datetime_format=True)

    # Fallback: yyyymmdd
    if pd.isna(d) and s2.isdigit() and len(s2) == 8:
        try:
            d = pd.to_datetime([s2], format="%Y%m%d", errors="coerce")[0]
        except Exception:
            d = pd.NaT

    return d

# 3) Apply to expected date columns and collect failures (for QC only)
DATE_COLS = ["Result Date", "_T_date", "_T1_date", "_T2_date", "_T3_date"]

date_fail_parts = []

for c in DATE_COLS:
    if c in df.columns:
        parsed = df[c].map(parse_one_date)
        # Keep just the date part (drop timezones/time)
        parsed = parsed.dt.normalize()

        # Collect any values that failed to parse (original non-null -> parsed NaT)
        mask_bad = df[c].notna() & parsed.isna()
        if mask_bad.any():
            bad = (df.loc[mask_bad, [c]]
                    .assign(Column=c, RowIndex=lambda x: x.index)
                    .rename(columns={c: "Original"}))
            date_fail_parts.append(bad[["RowIndex", "Column", "Original"]])

        df[c] = parsed

date_parse_failures = (pd.concat(date_fail_parts, ignore_index=True)
                       if date_fail_parts else
                       pd.DataFrame(columns=["RowIndex","Column","Original"]))

print("Date null counts after robust parsing:")
for c in DATE_COLS:
    if c in df.columns:
        print(f"  {c}: {int(df[c].isna().sum())} nulls")

if not date_parse_failures.empty:
    print("\nSample of unparsed date values (first 20):")
    display(date_parse_failures.head(20))
else:
    print("\nAll date values parsed successfully.")


C:\Users\cecme\AppData\Local\Temp\ipykernel_24996\2902613995.py:42: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  d = pd.to_datetime(s2, errors="coerce", dayfirst=True, infer_datetime_format=True)
C:\Users\cecme\AppData\Local\Temp\ipykernel_24996\2902613995.py:42: UserWarning: Parsing dates in %m-%d-%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d = pd.to_datetime(s2, errors="coerce", dayfirst=True, infer_datetime_format=True)
C:\Users\cecme\AppData\Local\Temp\ipykernel_24996\2902613995.py:42: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d = pd.to_datetime(s2, errors="coerce", dayfirst=True, inf

Date null counts after robust parsing:
  Result Date: 0 nulls
  _T_date: 1 nulls
  _T1_date: 1 nulls
  _T2_date: 1 nulls
  _T3_date: 1 nulls

All date values parsed successfully.


In [25]:
# Cell 4 — Coerce numeric columns (no feature engineering)
NUMERIC_CANDIDATES = [
    "Sales","Expenses","Operating Profit","OPM %","Other Income","Interest","Depreciation",
    "Profit before tax","Tax %","Net Profit","EPS in Rs",
    "Revenue","Financing Profit","Financing Margin %","Gross NPA %","Net NPA %",
    "Closing Price (T)","Closing Price (T+1)","Closing Price (T+2)","Closing Price (T+3)"
]

def _to_numeric_safe(series):
    if series.dtype != object and not pd.api.types.is_string_dtype(series):
        # already numeric-ish
        return pd.to_numeric(series, errors="coerce")
    return (
        series.astype(str)
              .str.replace(",", "", regex=False)
              .str.replace("%", "", regex=False)
              .str.replace("₹", "", regex=False)
              .str.replace("–", "-", regex=False)   # en-dash -> hyphen
              .replace({"None": np.nan, "nan": np.nan, "NaN": np.nan, "": np.nan})
              .pipe(pd.to_numeric, errors="coerce")
    )

for c in NUMERIC_CANDIDATES:
    if c in df.columns:
        df[c] = _to_numeric_safe(df[c])

print("Numeric coercion done. dtypes summary:")
print(df.dtypes)


Numeric coercion done. dtypes summary:
Company                                object
Ticker                                 object
AV_Ticker.BSE                          object
Sector                                 object
Cap Category                           object
                                    ...      
_T3_date                       datetime64[ns]
Closing Price (T+3)                   float64
3-Day Avg Price Post Result            object
3-Day Return (%)                       object
Movement Label (1,0)                   object
Length: 32, dtype: object


In [26]:
# Cell 5 — QC flags & summaries

def _flag_row_issues(row):
    flags = []

    # Missing Result Date
    if "Result Date" in row and pd.isna(row["Result Date"]):
        flags.append("MISSING_RESULT_DATE")

    # Duplicate key (Ticker + Result Date) — handled via join after we build a key series
    # We'll compute this outside row-wise.

    # T date after Result Date
    if "Result Date" in row and "_T_date" in row:
        rd, td = row["Result Date"], row["_T_date"]
        if pd.notna(rd) and pd.notna(td) and td > rd:
            flags.append("T_DATE_AFTER_RESULT")

    # Monotonicity of T..T+3 dates
    date_cols = ["_T_date","_T1_date","_T2_date","_T3_date"]
    dates = [row.get(c, pd.NaT) for c in date_cols if c in row.index]
    # remove NaT, then check nondecreasing
    dates_non_null = [d for d in dates if pd.notna(d)]
    if len(dates_non_null) > 1:
        if any(dates_non_null[i] > dates_non_null[i+1] for i in range(len(dates_non_null)-1)):
            flags.append("NON_ASCENDING_T_DATES")

    # Price/date pairing for T only
    if "_T_date" in row and "Closing Price (T)" in row:
        td, tp = row["_T_date"], row["Closing Price (T)"]
        if pd.notna(td) and pd.isna(tp):
            flags.append("MISSING_T_PRICE")
        if pd.isna(td) and pd.notna(tp):
            flags.append("MISSING_T_DATE")

    # Percent ranges (0..100) where applicable
    for pct_col in ["OPM %","Tax %","Financing Margin %","Gross NPA %","Net NPA %"]:
        if pct_col in row and pd.notna(row[pct_col]):
            val = row[pct_col]
            if not (0 <= val <= 100):
                flags.append(f"OUT_OF_RANGE_{pct_col.replace(' ','_')}")

    return flags

# Work on a defensive copy
df_qc = df.copy()

# Duplicate key detection (Ticker + Result Date)
key_cols = [c for c in ["Ticker","Result Date"] if c in df_qc.columns]
dup_key_series = pd.Series(False, index=df_qc.index)
if len(key_cols) == 2:
    dup_key_series = df_qc.duplicated(subset=key_cols, keep=False)

# Build flags
flag_list = []
for idx, row in df_qc.iterrows():
    row_flags = _flag_row_issues(row)
    if dup_key_series.loc[idx]:
        row_flags.append("DUPLICATE_KEY")
    if row_flags:
        flag_list.append({
            "RowIndex": idx,
            "Ticker": row.get("Ticker", np.nan),
            "Result Date": row.get("Result Date", pd.NaT),
            "Flags": ", ".join(sorted(set(row_flags)))
        })

qc_flags = pd.DataFrame(flag_list).sort_values(["Ticker","Result Date","RowIndex"])

# Flag summary
if not qc_flags.empty:
    # explode by comma to count each flag separately
    exploded = qc_flags.assign(Flag=qc_flags["Flags"].str.split(", ")).explode("Flag")
    qc_summary = exploded["Flag"].value_counts().rename_axis("Flag").reset_index(name="Count")
else:
    qc_summary = pd.DataFrame(columns=["Flag","Count"])

# Nulls by column
nulls_by_col = df.isna().sum().rename_axis("Column").reset_index(name="Nulls")

# Topline summary
summary_items = []
summary_items.append(("Rows", len(df)))
summary_items.append(("Unique Tickers", df["Ticker"].nunique() if "Ticker" in df.columns else np.nan))
summary_items.append(("Unique Sectors", df["Sector"].nunique() if "Sector" in df.columns else np.nan))
summary_items.append(("Missing Result Date", int(df["Result Date"].isna().sum() if "Result Date" in df.columns else 0)))
summary_items.append(("Missing _T_date", int(df["_T_date"].isna().sum() if "_T_date" in df.columns else 0)))
summary_items.append(("Missing Closing Price (T)", int(df["Closing Price (T)"].isna().sum() if "Closing Price (T)" in df.columns else 0)))
summary = pd.DataFrame(summary_items, columns=["Metric","Value"])

print("QC summary:")
print(qc_summary.head(20))
print("\nTopline summary:")
print(summary)


QC summary:
                              Flag  Count
0            NON_ASCENDING_T_DATES    114
1              T_DATE_AFTER_RESULT     67
2  OUT_OF_RANGE_Financing_Margin_%     61
3               OUT_OF_RANGE_Tax_%     11
4               OUT_OF_RANGE_OPM_%      1

Topline summary:
                      Metric  Value
0                       Rows    585
1             Unique Tickers     45
2             Unique Sectors      5
3        Missing Result Date      0
4            Missing _T_date      1
5  Missing Closing Price (T)      1


In [27]:
# Cell 6 — Save to Excel: clean_data + QC sheets (including date_parse_failures)

with pd.ExcelWriter(PATH_OUT, engine="openpyxl") as writer:
    # Cleaned data (same columns as input, parsed dates, trimmed text, numeric coercions from Cell 4)
    df.to_excel(writer, sheet_name="clean_data", index=False)

    # QC artifacts from Cell 5 (if you kept the same variable names)
    if 'qc_flags' in locals():       qc_flags.to_excel(writer, sheet_name="qc_flags", index=False)
    if 'qc_summary' in locals():     qc_summary.to_excel(writer, sheet_name="qc_summary", index=False)
    if 'nulls_by_col' in locals():   nulls_by_col.to_excel(writer, sheet_name="nulls_by_col", index=False)
    if 'summary' in locals():        summary.to_excel(writer, sheet_name="summary", index=False)

    # NEW: date parse failures (for diagnostics only)
    if 'date_parse_failures' in locals():
        date_parse_failures.to_excel(writer, sheet_name="date_parse_failures", index=False)

print(f"\nSaved cleaned workbook to: {PATH_OUT}")



Saved cleaned workbook to: MS_Capstone_Final_Quality_Check_Summary.xlsx
